Universal Meteorological Imputation Model
Trains a Bidirectional LSTM on multiple AWS node datasets with ERA5 auxiliary features,
cyclical time encodings, and a Node Embedding Layer.
Evaluates model on simulated missingness (10%-50%) and exports imputed datasets.


In [56]:
import os
import sys
import glob
import warnings
import asyncio
import logging

# Fix Windows Proactor event loop issue with ZMQ/TF
if sys.platform == 'win32':
    asyncio.set_event_loop_policy(asyncio.WindowsSelectorEventLoopPolicy())

warnings.filterwarnings('ignore')
os.environ['TF_ENABLE_ONEDNN_OPTS'] = '0'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import os
import numpy as np

# Safe Dynamic CuPy/NumPy Import (Pandas Compatible)
IS_KAGGLE = os.path.exists('/kaggle/input')
if IS_KAGGLE:
    try:
        import cupy as xp
        print("Using CuPy (xp) for GPU Array Operations (Kaggle Environment)")
    except ImportError:
        import numpy as xp
        print("CuPy not found, using NumPy as xp")
else:
    import numpy as xp
    print("Using NumPy (xp) for Array Operations (Local Environment)")
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from matplotlib.patches import Patch
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger()


Using NumPy (xp) for Array Operations (Local Environment)


# Environment & RUN_MODE Configuration


In [57]:
RUN_MODE = "LOCAL_TEST"  # Nilai yang tersedia: "LOCAL_TEST", "KAGGLE", "FULL_TRAIN"

# ==========================================
# BENCHMARK CONFIGURATION
# ==========================================
USE_ERA5_GUIDANCE = True

ACTIVE_MODELS = ["BiLSTM", "BiGRU", "CNN1D", "TCN", "AutoEncoder", "DenoisingAE", "TFT_Light", "SAITS_Light"]

# Deteksi otomatis apakah sistem berjalan di Kaggle Cloud
import os
IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ or os.path.exists('/kaggle/input')
if IS_KAGGLE:
    RUN_MODE = "FULL_TRAIN"
    print("[INFO] Kaggle environment detected. Enforcing KAGGLE mode.")
    BASE_DIR = '/kaggle/working'
    # Kaggle dataset paths specific to user's environment
    CACHE_DIR = '/kaggle/input/notebooks/jerismeteo/cek-data-sensor'
    PATH_ERA5 = '/kaggle/input/notebooks/jerismeteo/generate-era5-data/cuaca_gabungan_jerukagung.csv'
else:
    BASE_DIR = os.getcwd()
    BASE_DATA_DIR = os.path.abspath(os.path.join(BASE_DIR, '..'))
    CACHE_DIR = os.path.join(BASE_DATA_DIR, 'Analisis_Meteorologi', 'cache_data')
    PATH_ERA5 = os.path.join(BASE_DIR, 'cuaca_gabungan_jerukagung.csv')

OUTPUTS_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(OUTPUTS_DIR, exist_ok=True)

# Hyperparameter Tuning & Option Flags


In [58]:
RUN_TUNING = False  # Disabled for benchmarking

if RUN_MODE == "LOCAL_TEST":
    FIT_EPOCHS = 5
    FIT_BATCH_SIZE = 16
    FIT_VERBOSE = 1
elif RUN_MODE == "KAGGLE":
    FIT_EPOCHS = 50
    FIT_BATCH_SIZE = 128
    FIT_VERBOSE = 2
else:
    FIT_EPOCHS = 3
    FIT_BATCH_SIZE = 64
    FIT_VERBOSE = 1

SEQ_LEN = 120  # 120 jam = 5 hari
TARGET_VARS = ['temperature', 'humidity', 'pressure', 'dewpoint']


# TensorFlow & Keras Tuner Check


In [59]:
import subprocess
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers, callbacks as tf_callbacks
    print(f'[OK] TensorFlow v{tf.__version__} is active.')
except ImportError:
    print('[ERROR] TensorFlow/Keras is required for this Bidirectional LSTM model.')
    sys.exit(1)

try:
    import keras_tuner as kt
    print(f'[OK] Keras Tuner v{kt.__version__} is active.')
except ImportError:
    print('[INFO] Keras Tuner not found. Attempting to install...')
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "keras-tuner"])
        import keras_tuner as kt
        print(f'[OK] Keras Tuner v{kt.__version__} successfully installed and imported.')
    except Exception as e:
        print(f'[WARNING] Failed to install/import Keras Tuner: {e}')

from tensorflow.keras import mixed_precision
if RUN_MODE == 'KAGGLE':
    mixed_precision.set_global_policy('mixed_float16')
    print('[INFO] Mixed Precision enabled.')

[OK] TensorFlow v2.20.0 is active.
[OK] Keras Tuner v1.4.8 is active.


In [60]:
# ===========================================================
# Cek dan Inisialisasi TPU
# ===========================================================
try:
    # Mendeteksi TPU. Tidak ada biaya jika tidak ada TPU yang tersedia.
    tpu = tf.distribute.cluster_resolver.TPUClusterResolver.connect() 
    strategy = tf.distribute.TPUStrategy(tpu)
    logger.info(f'Running on TPU: {tpu.master()}')
    logger.info(f'{strategy.num_replicas_in_sync} replicas in sync.')
    # Sesuaikan batch size untuk TPU
    # TPU bekerja paling efisien dengan batch size besar yang merupakan kelipatan dari 8 * jumlah replica
    FIT_BATCH_SIZE = 8 * strategy.num_replicas_in_sync
    logger.info(f'Adjusted FIT_BATCH_SIZE for TPU: {FIT_BATCH_SIZE}')

except ValueError:
    # Jika TPU tidak ditemukan, gunakan strategy default (CPU atau GPU tunggal)
    logger.warning('TPU not found. Using default strategy (CPU/Single-GPU).')
    strategy = tf.distribute.get_strategy()

2026-06-28 18:20:30,913 - WARNING - TPU not found. Using default strategy (CPU/Single-GPU).


# 1. Ingest & Align AWS Node Data


In [61]:
print('\n[1/10] Scanning and loading AWS node datasets...')
node_files = glob.glob(os.path.join(CACHE_DIR, 'id-*_raw.csv'))
if not node_files:
    raise FileNotFoundError(f"No node CSV files matching 'id-*_raw.csv' found in {CACHE_DIR}")

node_data = {}
global_min_time = None
global_max_time = None

col_map = {
    'dew': 'dewpoint',
    'temperature': 'temperature',
    'humidity': 'humidity',
    'pressure': 'pressure'
}

for fpath in sorted(node_files):
    node_id = os.path.basename(fpath).split('_')[0]
    print(f"  Processing {node_id} from {os.path.basename(fpath)}...")
    
    df_raw = pd.read_csv(fpath)
    if 'timestamp' not in df_raw.columns:
        print(f"    [WARN] No timestamp column found in {node_id}. Skipping.")
        continue
        
    # Convert timestamp to local datetime
    df_raw['datetime'] = (pd.to_datetime(df_raw['timestamp'], unit='s', utc=True)
                          .dt.tz_convert('Asia/Jakarta')
                          .dt.tz_localize(None))
    
    df_raw = (df_raw.dropna(subset=['datetime'])
              .sort_values('datetime')
              .drop_duplicates(subset='datetime')
              .set_index('datetime'))
    
    # Rename key columns for standardization
    df_rename = df_raw.rename(columns=col_map)
    df_rename = df_rename.loc[:, ~df_rename.columns.duplicated()]
    
    # Meteorological Quality Control: Set outliers outside realistic tropical boundaries to NaN
    # This prevents sensor malfunction anomalies (e.g. pressure = -9 hPa) from corrupting the scale
    qc_bounds = {
        'temperature': (15.0, 45.0),
        'humidity': (40.0, 100.0),
        'pressure': (1002.0, 1019.0),
        'dewpoint': (15.0, 35.0)
    }
    for var, (vmin, vmax) in qc_bounds.items():
        if var in df_rename.columns:
            invalid_mask = (df_rename[var] < vmin) | (df_rename[var] > vmax)
            n_invalid = invalid_mask.sum()
            if n_invalid > 0:
                print(f"    [QC] {node_id} - {var}: replaced {n_invalid:,} outliers outside ({vmin}, {vmax}) with NaN")
                df_rename.loc[invalid_mask, var] = np.nan
    
    # Verify that all 4 target variables exist
    missing_cols = [c for c in TARGET_VARS if c not in df_rename.columns]
    if missing_cols:
        print(f"    [WARN] Targets {missing_cols} missing in {node_id}. Skipping.")
        continue
        
    # Resample to 1-minute mean
    df_1min = df_rename[TARGET_VARS].resample('1min').mean()
    
    # Track global temporal bounds
    node_min = df_1min.index.min()
    node_max = df_1min.index.max()
    if global_min_time is None or node_min < global_min_time:
        global_min_time = node_min
    if global_max_time is None or node_max > global_max_time:
        global_max_time = node_max
        
    node_data[node_id] = df_1min
    print(f"    Loaded {len(df_1min):,} minutes from {node_min} to {node_max}")

if not node_data:
    raise ValueError("No valid node datasets loaded.")

print(f"Global temporal bounds determined: {global_min_time} to {global_max_time}")

# Reindex all nodes to the global time range
global_time_range = pd.date_range(start=global_min_time, end=global_max_time, freq='1min', name='datetime')
if RUN_MODE == "LOCAL_TEST":
    global_time_range = global_time_range[-5760:]
print(f"Global aligned index length: {len(global_time_range):,} rows.")

for nid in list(node_data.keys()):
    node_data[nid] = node_data[nid].reindex(global_time_range)



[1/10] Scanning and loading AWS node datasets...
  Processing id-02 from id-02_raw.csv...
    Loaded 63,745 minutes from 2026-05-14 16:50:00 to 2026-06-27 23:14:00
  Processing id-03 from id-03_raw.csv...
    [QC] id-03 - pressure: replaced 1,191 outliers outside (1002.0, 1019.0) with NaN
    Loaded 1,338,002 minutes from 2023-12-11 19:13:00 to 2026-06-27 23:14:00
  Processing id-05 from id-05_raw.csv...
    [QC] id-05 - temperature: replaced 1,321 outliers outside (15.0, 45.0) with NaN
    [QC] id-05 - humidity: replaced 633 outliers outside (40.0, 100.0) with NaN
    Loaded 670,174 minutes from 2025-03-19 13:41:00 to 2026-06-27 23:14:00
Global temporal bounds determined: 2023-12-11 19:13:00 to 2026-06-27 23:14:00
Global aligned index length: 5,760 rows.


# 2. Ingest & Time-Interpolate ERA5 Auxiliary Data


In [62]:
print('\n[2/10] Loading and time-interpolating ERA5 auxiliary data...')
era5_raw = pd.read_csv(PATH_ERA5)
if 'datetime_utc' in era5_raw.columns:
    era5_raw.rename(columns={'datetime_utc': 'datetime'}, inplace=True)

era5_raw['datetime'] = (pd.to_datetime(era5_raw['datetime'], utc=True)
                        .dt.tz_convert('Asia/Jakarta')
                        .dt.tz_localize(None))
era5_raw = (era5_raw.sort_values('datetime')
            .drop_duplicates('datetime')
            .set_index('datetime'))

era5_req_cols = [
    'temperature_2m', 
    'relative_humidity_2m', 
    'dew_point_2m', 
    'pressure_msl',
    'surface_pressure',
    'wind_speed_10m', 
    'wind_direction_10m', 
    'cloud_cover', 
    'rain'
]
# Select columns or fallback if missing
era5_cols = [c for c in era5_req_cols if c in era5_raw.columns]
era5_feats = era5_raw[era5_cols]

# Temporal interpolation to 1-minute global time range
era5_reindexed = era5_feats.reindex(era5_feats.index.union(global_time_range)).sort_index()
era5_interpolated = era5_reindexed.interpolate(method='time')
era5_1min = era5_interpolated.reindex(global_time_range)
era5_1min.columns = [f'era5_{c}' for c in era5_1min.columns]

# Handle edge NaNs
era5_1min = era5_1min.ffill().bfill()
print(f"  ERA5 features processed: {list(era5_1min.columns)}")
print(f"  ERA5 NaNs remaining: {era5_1min.isna().sum().sum()}")


[2/10] Loading and time-interpolating ERA5 auxiliary data...
  ERA5 features processed: ['era5_temperature_2m', 'era5_relative_humidity_2m', 'era5_dew_point_2m', 'era5_pressure_msl', 'era5_surface_pressure', 'era5_wind_speed_10m', 'era5_wind_direction_10m', 'era5_cloud_cover', 'era5_rain']
  ERA5 NaNs remaining: 0


# 3. Create Cyclical Temporal Features


In [63]:
print('\n[3/10] Creating cyclical temporal features...')
time_feats = pd.DataFrame(index=global_time_range)
time_feats['hour_sin'] = np.sin(2 * np.pi * global_time_range.hour / 24)
time_feats['hour_cos'] = np.cos(2 * np.pi * global_time_range.hour / 24)
time_feats['doy_sin']  = np.sin(2 * np.pi * global_time_range.dayofyear / 366)
time_feats['doy_cos']  = np.cos(2 * np.pi * global_time_range.dayofyear / 366)
time_feats['month_sin'] = np.sin(2 * np.pi * global_time_range.month / 12)
time_feats['month_cos'] = np.cos(2 * np.pi * global_time_range.month / 12)



[3/10] Creating cyclical temporal features...


# 4. Phase 1: Linear Interpolation (Short Gaps <= 15 min)


In [64]:
print('\n[4/10] Running Phase 1: Linear interpolation on gaps <= 15 minutes...')
node_data_phase1 = {}
for nid, df_node in node_data.items():
    df_p1 = df_node.copy()
    for col in TARGET_VARS:
        before_nans = df_p1[col].isna().sum()
        df_p1[col] = df_p1[col].interpolate(method='pchip', limit=15, limit_direction='both')
        after_nans = df_p1[col].isna().sum()
        print(f"  {nid} - {col}: {before_nans:,} NaNs -> {after_nans:,} NaNs (filled {before_nans - after_nans:,})")
    node_data_phase1[nid] = df_p1



[4/10] Running Phase 1: Linear interpolation on gaps <= 15 minutes...
  id-02 - temperature: 73 NaNs -> 0 NaNs (filled 73)
  id-02 - humidity: 73 NaNs -> 0 NaNs (filled 73)
  id-02 - pressure: 73 NaNs -> 0 NaNs (filled 73)
  id-02 - dewpoint: 73 NaNs -> 0 NaNs (filled 73)
  id-03 - temperature: 8 NaNs -> 0 NaNs (filled 8)
  id-03 - humidity: 8 NaNs -> 0 NaNs (filled 8)
  id-03 - pressure: 8 NaNs -> 0 NaNs (filled 8)
  id-03 - dewpoint: 8 NaNs -> 0 NaNs (filled 8)
  id-05 - temperature: 26 NaNs -> 0 NaNs (filled 26)
  id-05 - humidity: 26 NaNs -> 0 NaNs (filled 26)
  id-05 - pressure: 26 NaNs -> 0 NaNs (filled 26)
  id-05 - dewpoint: 26 NaNs -> 0 NaNs (filled 26)


# 5. Denoising Autoencoder Setup (Inputs & Masks)


In [65]:
print('\n[5/10] Setting up Denoising Autoencoder inputs and masks...')
node_encoded_dfs = {}
for nid, df_node_p1 in node_data_phase1.items():
    df_node_orig = node_data[nid]
    df_setup = pd.DataFrame(index=global_time_range)
    
    # 1. Observation masks (1 if observed after Phase 1, 0 if missing)
    for col in TARGET_VARS:
        df_setup[f'mask_{col}'] = df_node_p1[col].notna().astype(np.float32)
        
    # 2. Temporary filled target features (fallback to corresponding ERA5 values)
    era5_map = {
        'temperature': 'era5_temperature_2m',
        'humidity': 'era5_relative_humidity_2m',
        'pressure': 'era5_pressure_msl',
        'dewpoint': 'era5_dew_point_2m'
    }
    
    for col in TARGET_VARS:
        # Fill missing values using the corresponding ERA5 proxy
        era_col = era5_map[col] if era5_map[col] in era5_1min.columns else era5_1min.columns[0]
        df_setup[f'filled_{col}'] = df_node_p1[col].fillna(era5_1min[era_col])
        
    # 3. Ground truth targets (keep original observed values before Phase 1 interpolation or after)
    # Note: Target for model optimization should be the original observed value
    for col in TARGET_VARS:
        df_setup[f'target_{col}'] = df_node_p1[col]
        # Weight for training: 1 if observed, 0 if missing
        df_setup[f'weight_{col}'] = df_node_p1[col].notna().astype(np.float32)
        
    # Combine with ERA5 and time features
    df_combined = pd.concat([df_setup, era5_1min, time_feats], axis=1)
    df_combined['node_id'] = nid
    
    # Forward/backward fill all features to prevent NaN features
    feature_cols = [c for c in df_combined.columns if not c.startswith('target_') and c != 'node_id']
    df_combined[feature_cols] = df_combined[feature_cols].ffill().bfill()
    cols_to_cast = [c for c in df_combined.columns if c != 'node_id']
    df_combined[cols_to_cast] = df_combined[cols_to_cast].astype('float32')
    
    node_encoded_dfs[nid] = df_combined

# Fit LabelEncoder for Node ID representation
le_node = LabelEncoder()
le_node.fit(list(node_data.keys()))
os.makedirs(os.path.join(OUTPUTS_DIR, 'model'), exist_ok=True)
joblib.dump(le_node, os.path.join(OUTPUTS_DIR, 'model', 'label_encoder_node.pkl'))
print(f"Node Encoder saved. Classes: {le_node.classes_}")

# Clean up phase 1 and era5 memory
import gc
del node_data_phase1
del era5_1min
del time_feats
gc.collect()



[5/10] Setting up Denoising Autoencoder inputs and masks...
Node Encoder saved. Classes: ['id-02' 'id-03' 'id-05']


151511

# 6. Extract Strided Sequences


> **💡 Catatan Arsitektur: Bagaimana SEQ_LEN=120 Menangkap Siklus (Cyclic)?**
> Walaupun `SEQ_LEN=120` pada data 1-menit hanya berarti **2 Jam**, model ini tetap memahami siklus harian (24-jam) dan tahunan.

Hal ini dimungkinkan karena pada *Step 3*, kita telah menginjeksi **Cyclical Temporal Features** (`hour_sin`, `hour_cos`, `doy_sin`, `doy_cos`).
Dengan adanya fitur ini, setiap menit data membawa koordinat waktu absolutnya sendiri, sehingga LSTM tidak perlu merekam sekuens sepanjang 4320 menit untuk 'menyadar' adanya pergantian siang dan malam. Ini menjaga memori (RAM) tetap rendah namun tingkat akurasi pola makro tetap tinggi.

In [66]:
print('\n[6/10] Extracting training and validation sequences...')
# Melihat 3 hari ke belakang (1440 menit/hari * 3 hari)
SEQ_LEN = 120

# Tetap membuat sampel pelatihan baru setiap jam untuk augmentasi data
STRIDE_TRAIN = 60

# Membuat sampel validasi yang tidak tumpang tindih
STRIDE_VAL = 120

sample_df = next(iter(node_encoded_dfs.values()))
continuous_features = [c for c in sample_df.columns if not c.startswith('target_') and not c.startswith('weight_') and c != 'node_id']
target_cols = [f'target_{col}' for col in TARGET_VARS]
weight_cols = [f'weight_{col}' for col in TARGET_VARS]

print(f"Continuous features count: {len(continuous_features)}")

split_point = int(0.8 * len(global_time_range))
train_range = global_time_range[:split_point]
val_range = global_time_range[split_point:]

# 1. First pass: count valid samples to preallocate memory
total_train = 0
total_val = 0
for nid, df_node in node_encoded_dfs.items():
    total_train += len(range(0, split_point - SEQ_LEN + 1, STRIDE_TRAIN))
    total_val += len(range(0, len(global_time_range) - split_point - SEQ_LEN + 1, STRIDE_VAL))

import numpy as np

# Pre-allocate arrays
X_train_cont = np.empty((total_train, SEQ_LEN, len(continuous_features)), dtype=np.float32)
X_train_node = np.empty((total_train, 1), dtype=np.int32)
Y_train = np.empty((total_train, SEQ_LEN, len(TARGET_VARS)), dtype=np.float32)
W_train = np.empty((total_train, SEQ_LEN, len(TARGET_VARS)), dtype=np.float32)

X_val_cont = np.empty((total_val, SEQ_LEN, len(continuous_features)), dtype=np.float32)
X_val_node = np.empty((total_val, 1), dtype=np.int32)
Y_val = np.empty((total_val, SEQ_LEN, len(TARGET_VARS)), dtype=np.float32)
W_val = np.empty((total_val, SEQ_LEN, len(TARGET_VARS)), dtype=np.float32)
X_val_list_times = []

# 2. Second pass: fill preallocated arrays
tr_idx = 0
vl_idx = 0

all_cols = continuous_features + target_cols + weight_cols
c_end = len(continuous_features)
t_end = c_end + len(target_cols)

# We use list to freeze dictionary keys so we can pop
nids = list(node_encoded_dfs.keys())
for nid in nids:
    node_idx = le_node.transform([nid])[0]
    
    # Extract arrays and free dataframe immediately!
    df_node = node_encoded_dfs[nid]
    
    df_train = df_node.loc[train_range]
    arr_train = df_train[all_cols].to_numpy(dtype=np.float32)
    cont_arr_train = arr_train[:, :c_end]
    target_arr_train = arr_train[:, c_end:t_end]
    weight_arr_train = arr_train[:, t_end:]
    
    for i in range(0, len(df_train) - SEQ_LEN + 1, STRIDE_TRAIN):
        seq_w = weight_arr_train[i:i+SEQ_LEN]
        if seq_w.sum() == 0:
            continue
        X_train_cont[tr_idx] = cont_arr_train[i:i+SEQ_LEN]
        X_train_node[tr_idx] = node_idx
        Y_train[tr_idx] = target_arr_train[i:i+SEQ_LEN]
        W_train[tr_idx] = seq_w
        tr_idx += 1
    
    # Free memory
    del df_train, arr_train, cont_arr_train, target_arr_train, weight_arr_train
    
    df_val = df_node.loc[val_range]
    arr_val = df_val[all_cols].to_numpy(dtype=np.float32)
    cont_arr_val = arr_val[:, :c_end]
    target_arr_val = arr_val[:, c_end:t_end]
    weight_arr_val = arr_val[:, t_end:]
    
    for i in range(0, len(df_val) - SEQ_LEN + 1, STRIDE_VAL):
        seq_w = weight_arr_val[i:i+SEQ_LEN]
        if seq_w.sum() == 0:
            continue
        X_val_cont[vl_idx] = cont_arr_val[i:i+SEQ_LEN]
        X_val_node[vl_idx] = node_idx
        Y_val[vl_idx] = target_arr_val[i:i+SEQ_LEN]
        W_val[vl_idx] = seq_w
        X_val_list_times.append(df_val.index[i:i+SEQ_LEN])
        vl_idx += 1
        
    # Free memory
    del df_val, arr_val, cont_arr_val, target_arr_val, weight_arr_val
    # del df_node
    
    import gc
    gc.collect()

# Truncate arrays to actual valid samples count
X_train_cont = X_train_cont[:tr_idx]
X_train_node = X_train_node[:tr_idx]
Y_train = Y_train[:tr_idx]
W_train = W_train[:tr_idx]

X_val_cont = X_val_cont[:vl_idx]
X_val_node = X_val_node[:vl_idx]
Y_val = Y_val[:vl_idx]
W_val = W_val[:vl_idx]
X_val_times = np.array(X_val_list_times)

print(f"Extracted Train samples: {X_train_cont.shape[0]:,}")
print(f"Extracted Val samples: {X_val_cont.shape[0]:,}")



[6/10] Extracting training and validation sequences...
Continuous features count: 23
Extracted Train samples: 225
Extracted Val samples: 27


# 7. Scalers & Normalization


In [67]:
print('\n[7/10] Normalizing features...')

# Normalization: Scalers fit on training data
scaler_X = MinMaxScaler()
num_features = X_train_cont.shape[2]
scaler_X.fit(X_train_cont.reshape(-1, num_features))



[7/10] Normalizing features...


MinMaxScaler()

## 1. Compute medians for target columns ignoring NaNs


In [68]:
# 1. Compute medians for target columns ignoring NaNs
Y_train_2d = Y_train.reshape(-1, len(TARGET_VARS))
medians = np.nanmedian(Y_train_2d, axis=0)


## 2. Construct clean target arrays by filling NaNs with column medians


In [69]:
# 2. Construct clean target arrays by filling NaNs with column medians
Y_train_clean_2d = np.copy(Y_train_2d)
for col_idx in range(len(TARGET_VARS)):
    nan_mask = np.isnan(Y_train_clean_2d[:, col_idx])
    Y_train_clean_2d[nan_mask, col_idx] = medians[col_idx]
Y_train_clean = Y_train_clean_2d.reshape(Y_train.shape)

Y_val_2d = Y_val.reshape(-1, len(TARGET_VARS))
Y_val_clean_2d = np.copy(Y_val_2d)
for col_idx in range(len(TARGET_VARS)):
    nan_mask = np.isnan(Y_val_clean_2d[:, col_idx])
    Y_val_clean_2d[nan_mask, col_idx] = medians[col_idx]
Y_val_clean = Y_val_clean_2d.reshape(Y_val.shape)


## 3. Fit scaler_Y on clean training data


In [70]:
# 3. Fit scaler_Y on clean training data
scaler_Y = MinMaxScaler()
scaler_Y.fit(Y_train_clean_2d)

# Save scalers
joblib.dump(scaler_X, os.path.join(OUTPUTS_DIR, 'model', 'scaler_X.pkl'))
joblib.dump(scaler_Y, os.path.join(OUTPUTS_DIR, 'model', 'scaler_Y.pkl'))

# Transform variables in-place to save memory
X_train_cont_s = scaler_X.transform(X_train_cont.reshape(-1, num_features)).reshape(X_train_cont.shape)
X_val_cont_s = scaler_X.transform(X_val_cont.reshape(-1, num_features)).reshape(X_val_cont.shape)

Y_train_s = scaler_Y.transform(Y_train_clean.reshape(-1, len(TARGET_VARS))).reshape(Y_train.shape)
Y_val_s = scaler_Y.transform(Y_val_clean.reshape(-1, len(TARGET_VARS))).reshape(Y_val.shape)

# Free unscaled arrays
# del X_train_cont, X_val_cont  # Disabled to keep unscaled data for missingness simulation
gc.collect()

print(f"Train samples scaled: {X_train_cont_s.shape[0]:,}")
print(f"Val samples scaled: {X_val_cont_s.shape[0]:,}")

# Pre-mask training set to simulate missingness dynamically
# For each training sample, mask a random rate of observations in the input features
for s in range(X_train_cont_s.shape[0]):
    # Random missingness rate
    rate = np.random.choice([0.1, 0.2, 0.3, 0.4, 0.5])
    # Identify timesteps where masks are 1 for this sequence (features columns 0-3 are masks)
    for col_idx in range(len(TARGET_VARS)):
        mask_feat_idx = col_idx  # masks are first 4 columns in features
        filled_feat_idx = len(TARGET_VARS) + col_idx # filled targets are next 4 columns
        
        observed_timesteps = np.where(X_train_cont_s[s, :, mask_feat_idx] == 1.0)[0]
        if len(observed_timesteps) > 0:
            num_to_mask = int(rate * len(observed_timesteps))
            mask_times = np.random.choice(observed_timesteps, size=num_to_mask, replace=False)
            
            # Mask features
            X_train_cont_s[s, mask_times, mask_feat_idx] = 0.0  # Set mask to 0
            
            # Fill with ERA5 corresponding feature scaled
            # ERA5 feature column index in features:
            # masks (4) + filled targets (4) + ERA5 (9) + time (6)
            # ERA5 variables start at index 8. Mapping:
            # temperature -> era5_temperature_2m (index 8)
            # humidity -> era5_relative_humidity_2m (index 9)
            # dewpoint -> era5_dew_point_2m (index 10)
            # pressure -> era5_pressure_msl (index 11)
            era_idx_map = {'temperature': 8, 'humidity': 9, 'dewpoint': 10, 'pressure': 11}
            era_col = era_idx_map[TARGET_VARS[col_idx]]
            X_train_cont_s[s, mask_times, filled_feat_idx] = X_train_cont_s[s, mask_times, era_col]


Train samples scaled: 225
Val samples scaled: 27


# 8. Build & Train Bidirectional LSTM Model


In [71]:
import json
import time

MODEL_CONFIGS = {
    'BiLSTM':      {'type': 'lstm', 'units1': 64, 'units2': 32, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'BiGRU':       {'type': 'gru',  'units1': 64, 'units2': 32, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'CNN1D':       {'type': 'cnn',  'filters1': 64, 'filters2': 32, 'kernel_size': 3, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'TCN':         {'type': 'tcn',  'filters': 64, 'kernel_size': 3, 'dilations': [1, 2, 4, 8], 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'AutoEncoder': {'type': 'ae',   'hidden': 64, 'bottleneck': 16, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'DenoisingAE': {'type': 'dae',  'hidden': 64, 'bottleneck': 16, 'noise': 0.1, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'TFT_Light':   {'type': 'tft',  'heads': 4, 'head_dim': 16, 'ff_dim': 64, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4},
    'SAITS_Light': {'type': 'saits','heads': 4, 'head_dim': 16, 'ff_dim': 64, 'dropout': 0.2, 'lr': 1e-3, 'node_emb': 4}
}

model_histories = {}
trained_models = {}
training_times = {}

# Filter dataset if ERA5 Guidance is disabled
X_train_final = X_train_cont_s.copy()
X_val_final = X_val_cont_s.copy()

actual_num_features = X_train_final.shape[2]

# TF.Data Pipeline for Kaggle Optimization
def make_tf_dataset(X_cont, X_node, Y, W, batch_size, is_training=True):
    dataset = tf.data.Dataset.from_tensor_slices(((X_cont, X_node), Y, W))
    if is_training:
        dataset = dataset.shuffle(buffer_size=1024)
    dataset = dataset.batch(batch_size)
    if RUN_MODE == 'KAGGLE':
        dataset = dataset.cache().prefetch(tf.data.AUTOTUNE)
    return dataset

train_dataset = make_tf_dataset(X_train_final, X_train_node, Y_train_s, W_train[:, :, 0], FIT_BATCH_SIZE, is_training=True)
val_dataset = make_tf_dataset(X_val_final, X_val_node, Y_val_s, W_val[:, :, 0], FIT_BATCH_SIZE, is_training=False)

with strategy.scope():
    for model_name in ACTIVE_MODELS:
        if model_name not in MODEL_CONFIGS: continue
        cfg = MODEL_CONFIGS[model_name]
        
        print(f"\n{'='*60}")
        print(f"[INFO] Pelatihan Model: {model_name}...")
        print(f"{'='*60}")
        
        m_out_dir = os.path.join(OUTPUTS_DIR, model_name)
        for d in ['model', 'plots', 'metrics', 'predictions', 'reports', 'checkpoints']:
            os.makedirs(os.path.join(m_out_dir, d), exist_ok=True)
            
        feat_in = keras.Input(shape=(SEQ_LEN, actual_num_features), name='continuous_input')
        node_in = keras.Input(shape=(1,), name='node_input')
        
        num_nodes = len(le_node.classes_)
        node_emb = layers.Embedding(input_dim=num_nodes, output_dim=cfg['node_emb'], name='node_embedding')(node_in)
        node_emb = layers.Reshape((cfg['node_emb'],))(node_emb)
        node_emb_rep = layers.RepeatVector(SEQ_LEN)(node_emb)
        merged = layers.Concatenate(axis=-1)([feat_in, node_emb_rep])
        
        x = merged
        if cfg['type'] == 'lstm':
            x = layers.Bidirectional(layers.LSTM(cfg['units1'], return_sequences=True))(x)
            x = layers.Dropout(cfg['dropout'])(x)
            x = layers.Bidirectional(layers.LSTM(cfg['units2'], return_sequences=True))(x)
            
        elif cfg['type'] == 'gru':
            x = layers.Bidirectional(layers.GRU(cfg['units1'], return_sequences=True))(x)
            x = layers.Dropout(cfg['dropout'])(x)
            x = layers.Bidirectional(layers.GRU(cfg['units2'], return_sequences=True))(x)
            
        elif cfg['type'] == 'cnn':
            x = layers.Conv1D(filters=cfg['filters1'], kernel_size=cfg['kernel_size'], padding='same', activation='relu')(x)
            x = layers.Dropout(cfg['dropout'])(x)
            x = layers.Conv1D(filters=cfg['filters2'], kernel_size=cfg['kernel_size'], padding='same', activation='relu')(x)
            
        elif cfg['type'] == 'tcn':
            for d in cfg['dilations']:
                skip = x
                x = layers.Conv1D(filters=cfg['filters'], kernel_size=cfg['kernel_size'], padding='causal', dilation_rate=d, activation='relu')(x)
                x = layers.Dropout(cfg['dropout'])(x)
                if skip.shape[-1] != x.shape[-1]:
                    skip = layers.Conv1D(filters=cfg['filters'], kernel_size=1, padding='same')(skip)
                x = layers.Add()([skip, x])
                
        elif cfg['type'] == 'ae':
            # Encoder
            x = layers.TimeDistributed(layers.Dense(cfg['hidden'], activation='relu'))(x)
            x = layers.TimeDistributed(layers.Dense(cfg['bottleneck'], activation='relu', name='latent_space'))(x)
            # Decoder
            x = layers.TimeDistributed(layers.Dense(cfg['hidden'], activation='relu'))(x)
            
        elif cfg['type'] == 'dae':
            # Noise Injection
            x = layers.GaussianNoise(cfg['noise'])(x)
            # Encoder
            x = layers.TimeDistributed(layers.Dense(cfg['hidden'], activation='relu'))(x)
            x = layers.TimeDistributed(layers.Dense(cfg['bottleneck'], activation='relu', name='latent_space'))(x)
            # Decoder
            x = layers.TimeDistributed(layers.Dense(cfg['hidden'], activation='relu'))(x)
            
        elif cfg['type'] in ['tft', 'saits']:
            # Simplified Attention
            x = layers.Dense(cfg['head_dim'] * cfg['heads'])(x)
            attn_out = layers.MultiHeadAttention(num_heads=cfg['heads'], key_dim=cfg['head_dim'])(x, x)
            x = layers.Add()([x, attn_out])
            x = layers.LayerNormalization()(x)
            ff_out = layers.Dense(cfg['ff_dim'], activation='relu')(x)
            ff_out = layers.Dense(cfg['head_dim'] * cfg['heads'])(ff_out)
            x = layers.Add()([x, ff_out])
            x = layers.LayerNormalization()(x)

        # Final projection to targets
        out = layers.TimeDistributed(layers.Dense(len(TARGET_VARS), activation='sigmoid'), name='imputation_output')(x)
        
        # Enforce float32 for output layer if using mixed precision
        if RUN_MODE == 'KAGGLE':
            out = layers.Activation('linear', dtype='float32')(out)
            
        model = keras.Model(inputs=[feat_in, node_in], outputs=out, name=f'Imputer_{model_name}')
        model.compile(optimizer=keras.optimizers.Adam(cfg['lr']), loss='mse', metrics=['mae'])
        
        cb_list = [
            tf_callbacks.EarlyStopping(patience=8, restore_best_weights=True, monitor='val_loss'),
            tf_callbacks.ReduceLROnPlateau(patience=4, factor=0.5, monitor='val_loss'),
            tf_callbacks.ModelCheckpoint(os.path.join(m_out_dir, 'checkpoints', 'best_model.keras'), save_best_only=True, monitor='val_loss')
        ]
        
        start_time = time.time()
        history = model.fit(
            train_dataset,
            validation_data=val_dataset,
            epochs=FIT_EPOCHS,
            callbacks=cb_list,
            verbose=FIT_VERBOSE
        )
        end_time = time.time()
        training_times[model_name] = end_time - start_time
        
        trained_models[model_name] = model
        model_histories[model_name] = history
        
        # Save training history JSON
        with open(os.path.join(m_out_dir, 'metrics', 'training_history.json'), 'w') as f:
            json.dump(history.history, f)
            
        # Plot Loss Curve
        plt.figure(figsize=(10, 4))
        plt.plot(history.history['loss'], label='Train Loss')
        plt.plot(history.history['val_loss'], label='Val Loss')
        plt.title(f'Kurva Pelatihan - {model_name}')
        plt.xlabel('Epoch')
        plt.ylabel('Loss (MSE)')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(m_out_dir, 'plots', 'loss_curve.png'), dpi=120)
        plt.close()


[INFO] Pelatihan Model: BiLSTM...
Epoch 1/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 53s 2s/step - loss: 0.0320 - mae: 0.1452 - val_loss: 0.0188 - val_mae: 0.1112 - learning_rate: 0.0010
Epoch 2/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 404ms/step - loss: 0.0131 - mae: 0.0915 - val_loss: 0.0071 - val_mae: 0.0668 - learning_rate: 0.0010
Epoch 3/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 7s 436ms/step - loss: 0.0071 - mae: 0.0664 - val_loss: 0.0053 - val_mae: 0.0584 - learning_rate: 0.0010
Epoch 4/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 22s 2s/step - loss: 0.0043 - mae: 0.0506 - val_loss: 0.0036 - val_mae: 0.0471 - learning_rate: 0.0010
Epoch 5/5
15/15 ━━━━━━━━━━━━━━━━━━━━ 24s 2s/step - loss: 0.0032 - mae: 0.0432 - val_loss: 0.0028 - val_mae: 0.0411 - learning_rate: 0.0010

[INFO] Pelatihan Model: BiGRU...
Epoch 1/5
 9/15 ━━━━━━━━━━━━━━━━━━━━ 1s 264ms/step - loss: 0.0504 - mae: 0.1838

KeyboardInterrupt: 

# 9. Multi-Rate Simulated Missingness Evaluation


In [ ]:
print('\n[9/10] Evaluasi Hasil Imputasi pada Gaps Sintetis (10%-50%)...')
from sklearn.metrics import r2_score

rates = [0.1, 0.2, 0.3, 0.4, 0.5]
all_models_metrics = {}

def calculate_nse(y_true, y_pred):
    return 1 - (np.sum((y_true - y_pred) ** 2) / (np.sum((y_true - np.mean(y_true)) ** 2) + 1e-8))

def calculate_kge(y_true, y_pred):
    r = np.corrcoef(y_true, y_pred)[0, 1] if len(y_true) > 1 else 0
    alpha = np.std(y_pred) / (np.std(y_true) + 1e-8)
    beta = np.mean(y_pred) / (np.mean(y_true) + 1e-8)
    return 1 - np.sqrt((r - 1)**2 + (alpha - 1)**2 + (beta - 1)**2)

def evaluate_imputation(y_true, y_pred, mask):
    yt = y_true[mask]
    yp = y_pred[mask]
    
    # Filter out NaNs (original ground truth might already be missing at the simulated mask locations)
    valid_mask = ~np.isnan(yt) & ~np.isnan(yp)
    yt = yt[valid_mask]
    yp = yp[valid_mask]
    
    if len(yt) < 2:
        return {'RMSE': 0, 'MAE': 0, 'R2': 0, 'NSE': 0, 'KGE': 0, 'Bias': 0, 'Correlation': 0}
    
    rmse = np.sqrt(mean_squared_error(yt, yp))
    mae = mean_absolute_error(yt, yp)
    r2 = r2_score(yt, yp)
    nse = calculate_nse(yt, yp)
    kge = calculate_kge(yt, yp)
    bias = np.mean(yp - yt)
    corr = np.corrcoef(yt, yp)[0, 1] if np.std(yt) > 0 and np.std(yp) > 0 else 0
    
    return {'RMSE': rmse, 'MAE': mae, 'R2': r2, 'NSE': nse, 'KGE': kge, 'Bias': bias, 'Correlation': corr}

with strategy.scope():
    for model_name, model in trained_models.items():
        print(f"\n--- Menguji {model_name} ---")
        m_out_dir = os.path.join(OUTPUTS_DIR, model_name)
        val_metrics = []
        
        for r in rates:
            print(f"  Missingness {int(r*100)}%...", end='')
            X_val_sim = X_val_cont.copy()
            Y_val_sim = Y_val.copy()
            
            mask_sim = np.random.rand(*Y_val_sim.shape) < r
            x_targets = X_val_sim[:, :, :len(TARGET_VARS)]
            x_targets[mask_sim] = np.nan
            X_val_sim[:, :, :len(TARGET_VARS)] = x_targets
            Y_val_sim[mask_sim] = np.nan
            
            # Normalisasi
            X_val_sim_s = scaler_X.transform(X_val_sim.reshape(-1, num_features)).reshape(X_val_sim.shape)
            
            # Filter ERA5 if disabled
            if not USE_ERA5_GUIDANCE:
                X_val_sim_s = X_val_sim_s[:, :, :len(TARGET_VARS)]
                
            for i_col in range(X_val_sim_s.shape[2]):
                m_v = np.nanmedian(X_val_sim_s[:, :, i_col])
                if np.isnan(m_v): m_v = 0
                np.nan_to_num(X_val_sim_s[:, :, i_col], nan=m_v, copy=False)
                
            preds_s = model.predict([X_val_sim_s, X_val_node], batch_size=FIT_BATCH_SIZE, verbose=0)
            preds = scaler_Y.inverse_transform(preds_s.reshape(-1, len(TARGET_VARS))).reshape(preds_s.shape)
            
            res = {'Missingness': f'{int(r*100)}%'}
            for col_idx, col_name in enumerate(TARGET_VARS):
                m = evaluate_imputation(Y_val[:, :, col_idx].flatten(), preds[:, :, col_idx].flatten(), mask_sim[:, :, col_idx].flatten())
                for k, v in m.items():
                    res[f'{col_name} {k}'] = v
            val_metrics.append(res)
            print(" Selesai.")
            
        df_metrics = pd.DataFrame(val_metrics)
        df_metrics.to_csv(os.path.join(m_out_dir, 'metrics', 'metrics_missingness.csv'), index=False)
        all_models_metrics[model_name] = df_metrics



[9/10] Evaluasi Hasil Imputasi pada Gaps Sintetis (10%-50%)...

--- Menguji BiLSTM ---
  Missingness 10%... Selesai.
  Missingness 20%... Selesai.
  Missingness 30%... Selesai.
  Missingness 40%... Selesai.
  Missingness 50%... Selesai.

--- Menguji BiGRU ---
  Missingness 10%... Selesai.
  Missingness 20%... Selesai.
  Missingness 30%... Selesai.
  Missingness 40%... Selesai.
  Missingness 50%... Selesai.

--- Menguji CNN1D ---
  Missingness 10%... Selesai.
  Missingness 20%... Selesai.
  Missingness 30%... Selesai.
  Missingness 40%... Selesai.
  Missingness 50%... Selesai.

--- Menguji TCN ---
  Missingness 10%... Selesai.
  Missingness 20%... Selesai.
  Missingness 30%... Selesai.
  Missingness 40%... Selesai.
  Missingness 50%... Selesai.

--- Menguji AutoEncoder ---
  Missingness 10%... Selesai.
  Missingness 20%... Selesai.
  Missingness 30%... Selesai.
  Missingness 40%... Selesai.
  Missingness 50%... Selesai.

--- Menguji DenoisingAE ---
  Missingness 10%... Selesai.
  Missi

# 10. Gap Filling (Inference) and Exporting Final CSVs


In [ ]:
print('\n[10/10] Rekonstruksi Gaps Aktual (Prediksi Penuh)...')

with strategy.scope():
    for model_name, model in trained_models.items():
        print(f"\n--- Prediksi dengan {model_name} ---")
        m_out_dir = os.path.join(OUTPUTS_DIR, model_name)
        
        for nid, df_node_raw in node_data.items():
            
            df_encoded = node_encoded_dfs[nid]
            
            cont_arr = df_encoded[continuous_features].values
            indices = list(range(0, len(cont_arr) - SEQ_LEN + 1, 1))
            
            X_full_list = []
            for i in indices:
                X_full_list.append(cont_arr[i:i+SEQ_LEN])
            
            import numpy as np
            X_full = np.array(X_full_list, dtype=np.float32)
            node_idx = le_node.transform([nid])[0]
            X_node = np.full((X_full.shape[0], 1), node_idx)
            
            X_full_s = scaler_X.transform(X_full.reshape(-1, num_features)).reshape(X_full.shape)
            if not USE_ERA5_GUIDANCE:
                X_full_s = X_full_s[:, :, :len(TARGET_VARS)]
                
            for i in range(X_full_s.shape[2]):
                m_v = np.nanmedian(X_full_s[:, :, i])
                if np.isnan(m_v): m_v = 0
                np.nan_to_num(X_full_s[:, :, i], nan=m_v, copy=False)
                
            preds_s = model.predict([X_full_s, X_node], batch_size=FIT_BATCH_SIZE, verbose=0)
            preds = scaler_Y.inverse_transform(preds_s.reshape(-1, len(TARGET_VARS))).reshape(preds_s.shape)
            
            pred_sum = np.zeros_like(df_node_raw[TARGET_VARS].values, dtype=np.float64)
            pred_count = np.zeros_like(df_node_raw[TARGET_VARS].values, dtype=np.float64)
            
            for i, start_idx in enumerate(indices):
                pred_sum[start_idx : start_idx + SEQ_LEN] += preds[i]
                pred_count[start_idx : start_idx + SEQ_LEN] += 1
                
            final_preds = np.divide(pred_sum, pred_count, out=np.zeros_like(pred_sum), where=pred_count!=0)
            
            df_imputed = df_node_raw.copy()
            for col_idx, col_name in enumerate(TARGET_VARS):
                mask_missing = df_imputed[col_name].isna()
                df_imputed.loc[mask_missing, col_name] = final_preds[mask_missing, col_idx]
            
            df_imputed.to_csv(os.path.join(m_out_dir, 'predictions', f'{nid}_imputed.csv'), index=True)


[10/10] Rekonstruksi Gaps Aktual (Prediksi Penuh)...

--- Prediksi dengan BiLSTM ---

--- Prediksi dengan BiGRU ---

--- Prediksi dengan CNN1D ---

--- Prediksi dengan TCN ---

--- Prediksi dengan AutoEncoder ---

--- Prediksi dengan DenoisingAE ---

--- Prediksi dengan TFT_Light ---

--- Prediksi dengan SAITS_Light ---


# 11. Plot Comparison of Imputed Gaps for Each Node


In [ ]:
print('\n[11/10] Visualisasi Plot Evaluasi...')

for model_name in trained_models.keys():
    m_out_dir = os.path.join(OUTPUTS_DIR, model_name)
    
    # Ambil 1 Node sebagai perwakilan visualisasi
    nid = list(node_data.keys())[0]
    out_path = os.path.join(m_out_dir, 'predictions', f'{nid}_imputed.csv')
    if not os.path.exists(out_path): continue
    
    df_imp = pd.read_csv(out_path)
    df_raw = node_data[nid]
    if 'datetime' in df_imp.columns:
        df_imp['datetime'] = pd.to_datetime(df_imp['datetime'], utc=True)
    else:
        df_imp = df_imp.reset_index()
        if 'datetime' in df_imp.columns:
            df_imp['datetime'] = pd.to_datetime(df_imp['datetime'], utc=True)
    if 'datetime' in df_raw.columns:
        df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], utc=True)
    else:
        df_raw = df_raw.reset_index()
        if 'datetime' in df_raw.columns:
            df_raw['datetime'] = pd.to_datetime(df_raw['datetime'], utc=True)
    
    for target in TARGET_VARS:
        mask_missing = df_raw[target].isna()
        y_obs = df_raw[target][~mask_missing]
        y_imp = df_imp[target][~mask_missing] # Untuk scatter, bandingkan yg ada
        
        # Scatter Plot: Observed vs Imputed
        plt.figure(figsize=(6,6))
        plt.scatter(y_obs, y_imp, alpha=0.3, color='teal')
        max_val = max(y_obs.max(), y_imp.max())
        plt.plot([0, max_val], [0, max_val], color='red', linestyle='--')
        plt.title(f'Observasi vs Imputasi ({target}) - {model_name}')
        plt.xlabel('Observasi Aktual')
        plt.ylabel('Hasil Imputasi')
        plt.tight_layout()
        plt.savefig(os.path.join(m_out_dir, 'plots', f'scatter_{target}.png'), dpi=120)
        plt.close()
        
        # Residual Histogram
        residuals = y_obs - y_imp
        plt.figure(figsize=(6,4))
        plt.hist(residuals.dropna(), bins=50, color='crimson', alpha=0.7)
        plt.title(f'Distribusi Residual ({target}) - {model_name}')
        plt.xlabel('Error (Obs - Imp)')
        plt.tight_layout()
        plt.savefig(os.path.join(m_out_dir, 'plots', f'residual_{target}.png'), dpi=120)
        plt.close()
        
        # Time Series Comparison (Zoom in 500 hours)
        plt.figure(figsize=(15, 5))
        plt.plot(df_imp['datetime'][:500], df_imp[target][:500], color='red', alpha=0.8, label='Imputed', linewidth=1.5)
        plt.plot(df_raw['datetime'][:500], df_raw[target][:500], color='blue', alpha=0.6, label='Observed AWS', linewidth=2)
        if USE_ERA5_GUIDANCE and f'{target}_era5' in df_imp.columns:
             plt.plot(df_imp['datetime'][:500], df_imp[f'{target}_era5'][:500], color='green', alpha=0.5, label='ERA5', linestyle='--')
        plt.title(f'Perbandingan Runtun Waktu ({target}) - {model_name}')
        plt.legend()
        plt.tight_layout()
        plt.savefig(os.path.join(m_out_dir, 'plots', f'timeseries_{target}.png'), dpi=120)
        plt.close()


[11/10] Visualisasi Plot Evaluasi...


In [ ]:
print('\n[12/10] Perbandingan Antar Model (Leaderboard)...')

cmp_dir = os.path.join(OUTPUTS_DIR, 'model_comparison')
os.makedirs(cmp_dir, exist_ok=True)

leaderboard_data = []

for model_name, df in all_models_metrics.items():
    avg_rmse = df[[c for c in df.columns if 'RMSE' in c]].mean().mean()
    avg_nse = df[[c for c in df.columns if 'NSE' in c]].mean().mean()
    avg_kge = df[[c for c in df.columns if 'KGE' in c]].mean().mean()
    
    leaderboard_data.append({
        'Model': model_name,
        'Average RMSE': avg_rmse,
        'Average NSE': avg_nse,
        'Average KGE': avg_kge,
        'Training Time (s)': training_times.get(model_name, 0)
    })

# Sort by NSE (higher is better) or RMSE (lower is better)
df_lb = pd.DataFrame(leaderboard_data).sort_values(by='Average RMSE', ascending=True)
df_lb.to_csv(os.path.join(cmp_dir, 'leaderboard.csv'), index=False)
df_lb.to_json(os.path.join(cmp_dir, 'leaderboard.json'), orient='records')

print(df_lb)

# Error Boxplot Comparison (Across all variables and missing rates)
plt.figure(figsize=(10, 6))
plot_data = []
labels = []
for model_name, df in all_models_metrics.items():
    rmse_cols = [c for c in df.columns if 'RMSE' in c]
    plot_data.append(df[rmse_cols].values.flatten())
    labels.append(model_name)
    
plt.boxplot(plot_data, labels=labels)
plt.title('Distribusi RMSE Antar Model (Semua Variabel & % Kosong)')
plt.ylabel('RMSE')
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.savefig(os.path.join(cmp_dir, 'error_boxplot_comparison.png'), dpi=120)
plt.close()

print(f"Leaderboard dan komparasi berhasil disimpan ke {cmp_dir}")
# Generate Comprehensive Summary Report
reports_dir = os.path.join(OUTPUTS_DIR, 'reports')
os.makedirs(reports_dir, exist_ok=True)
comprehensive_report = {}
for m_name, m_df in all_models_metrics.items():
    comprehensive_report[m_name] = {
        'training_time_s': training_times.get(m_name, 0),
        'metrics_details': m_df.to_dict(orient='records')
    }

import json
with open(os.path.join(reports_dir, 'all_models_summary.json'), 'w') as f:
    json.dump(comprehensive_report, f, indent=4)

print(f"Comprehensive report saved to {reports_dir}")



[12/10] Perbandingan Antar Model (Leaderboard)...
         Model  Average RMSE  Average NSE  Average KGE  Training Time (s)
0       BiLSTM      1.577556     0.532529     0.703219          12.075744
3          TCN      1.652614     0.425862     0.681884           8.052840
1        BiGRU      1.675857     0.517170     0.658671          13.265051
6    TFT_Light      1.900997     0.283400     0.622645          11.069639
2        CNN1D      2.169136     0.258854     0.226699           6.154610
7  SAITS_Light      2.346835     0.090867     0.614547          10.152905
4  AutoEncoder      2.425864     0.047131     0.109603          30.529866
5  DenoisingAE      2.627168    -0.059961    -0.070249          25.462792
Leaderboard dan komparasi berhasil disimpan ke d:\Github\Projek_Rainfall\Model_Imputasi\outputs\model_comparison
Comprehensive report saved to d:\Github\Projek_Rainfall\Model_Imputasi\outputs\reports
